In [13]:
import pandas as pd
import numpy as np
rng = np.random.default_rng(42)

# Load data
phys = pd.read_csv("physiological_cycles.csv")
slp  = pd.read_csv("sleeps.csv")
wkt  = pd.read_csv("workouts.csv")

# Convert timestamps to datetimes and extract a daily key
# For physiological and sleep data, use "Cycle start time" as the reference
phys["Cycle start time"] = pd.to_datetime(phys["Cycle start time"], errors="coerce")
slp["Cycle start time"]  = pd.to_datetime(slp["Cycle start time"],  errors="coerce")

phys["day"] = phys["Cycle start time"].dt.date
slp["day"]  = slp["Cycle start time"].dt.date

# For workouts, use "Workout start time" as the reference
wkt["Workout start time"] = pd.to_datetime(wkt["Workout start time"], errors="coerce")
wkt["day"] = wkt["Workout start time"].dt.date

# Create a sorted list
# Which is going to be the index for all parameters
unique_days = sorted(set(phys["day"]) | set(slp["day"]) | set(wkt["day"]))

# steps: daily step count.
# autogenerated
steps = rng.normal(10000, 2500, len(unique_days)).clip(3000, 20000)

# tib: "time in bed" per day in HOURS.
# "In bed duration (min)" summed per day.
tib = (
    slp.groupby("day")["In bed duration (min)"]
       .sum()
       .reindex(unique_days, fill_value=0)
       .to_numpy() / 60.0
)

# ac: "active calories burned" per day in CALORIES.
# "Energy burned (kcal)" summed per day.
ac = (
    phys.groupby("day")["Energy burned (cal)"]
        .sum()
        .reindex(unique_days, fill_value=0)
        .to_numpy() / 1000.0
)

# exm: "exercise minutes" per day in MINUTES.
# "Duration (min)" for exercise summed per day.
exm = (
    wkt.groupby("day")["Duration (min)"]
       .sum()
       .reindex(unique_days, fill_value=0)
       .to_numpy()
)

# nsed: "non-sedentary hours" per day in HOURS.
# total hours in a day minus time spent in be
nsed = 24 - tib

# dlh: "daylight hours" per day in HOURS.
# autogenerated
# correlate slightly with non-sedentary hours (nsed)
dlh = (rng.normal(1.5, 0.7, len(unique_days)) + (nsed - 14) * 0.1).clip(0.5, 4.0)

# dist: "distance walked" per day in METERS.
# autogenerated
# linked to steps (avg step ≈ 0.78m)
dist = (steps * rng.normal(0.75, 0.05, len(unique_days))).clip(1000, 20000)

# flt: "flights of stairs climbed" per day (COUNT).
# autogenerated
# loosely based on intensity of the workout
flt = rng.normal(8, 6, len(unique_days)).clip(0, 30)

# rhr: "average resting heart rate" per day in BPM.
# "Resting heart rate (bpm)".
rhr = (
    phys.groupby("day")["Resting heart rate (bpm)"]
        .mean()
        .reindex(unique_days, fill_value=0)
        .to_numpy()
)

In [14]:
# Steps, daylight hours, distance walked, flights of stairs climbed is missing
# non-sedentary hours is (24 - time in bed) because we don't have that data

In [15]:
steps = steps.astype(int)
dist = dist.astype(int)
flt = np.round(flt).astype(int)

X_raw = np.vstack([steps, tib, ac, exm, nsed, dlh, dist, flt, rhr]).T
X_raw = np.nan_to_num(X_raw, nan=0.0, posinf=0.0, neginf=0.0)

# phys has per-cycle rows; assume you already aggregated by day
recov = (phys.groupby("day")["Recovery score %"].mean().reindex(unique_days, fill_value=np.nan))
sleep_perf = (phys.groupby("day")["Sleep performance %"].mean().reindex(unique_days, fill_value=np.nan))
strain = (phys.groupby("day")["Day Strain"].mean().reindex(unique_days, fill_value=np.nan))

# Heuristic: high recovery + good sleep + moderate strain
valid_days_mask = ~(np.isnan(recov) | np.isnan(sleep_perf) | np.isnan(strain))
recov_clean = recov[valid_days_mask].to_numpy()
sleep_perf_clean = sleep_perf[valid_days_mask].to_numpy()
strain_clean = strain[valid_days_mask].to_numpy()
X_raw_clean = X_raw[valid_days_mask]

raw_score = (0.5 * recov_clean/100 + 0.3 * sleep_perf_clean/100 - 0.2 * ((strain_clean - 10)/10))
raw_score = np.clip(raw_score, 0, 1)

# Scale to use full range [0,10] based on actual data
raw_min, raw_max = raw_score.min(), raw_score.max()
raw_score_scaled = (raw_score - raw_min) / (raw_max - raw_min)
y_score = (raw_score_scaled * 10.0).reshape(-1, 1)

print(f"Using {len(y_score)} days with complete physiology data")
print(f"y_score range: [{y_score.min():.2f}, {y_score.max():.2f}]")

Using 564 days with complete physiology data
y_score range: [0.00, 10.00]


In [16]:
X_mean = X_raw_clean.mean(0, keepdims=True)
X_std = X_raw_clean.std(0, keepdims=True)
X_std[X_std < 1e-8] = 1.0
X_norm = (X_raw_clean - X_mean) / X_std

# Train linear regression model
Φ = np.hstack([X_norm, np.ones((X_norm.shape[0], 1))])
y = y_score / 10.0  # Scale to [0,1] for training

w = np.zeros((Φ.shape[1], 1))
η = 0.001
λ = 1e-5  # Reduced regularization

def f(Φ, y, w, λ):
    return np.mean((Φ @ w - y)**2) + λ * (w.T @ w)

def ߜf(Φ, y, w, λ):
    return (2/Φ.shape[0]) * (Φ.T @ (Φ @ w - y)) + 2 * λ * w

# Train the model
for i in range(2000):
    w = w - η * ߜf(Φ, y, w, λ)
    if i % 500 == 0:
        train_pred = (Φ @ w) * 10.0
        current_loss = f(Φ, y, w, λ)
        print(f"Iter {i}: Loss = {current_loss[0,0]:.6f}, "
              f"Max prediction = {np.max(train_pred):.2f}")

# Check model performance
train_predictions = (Φ @ w) * 10.0
print(f"Training predictions range: [{train_predictions.min():.2f}, {train_predictions.max():.2f}]")
print(f"Actual y_score range: [{y_score.min():.2f}, {y_score.max():.2f}]")

Iter 0: Loss = 0.346169, Max prediction = 0.02
Iter 500: Loss = 0.068034, Max prediction = 5.03
Iter 1000: Loss = 0.030186, Max prediction = 6.89
Iter 1500: Loss = 0.024801, Max prediction = 7.60
Training predictions range: [-2.08, 7.89]
Actual y_score range: [0.00, 10.00]


In [17]:
low = np.array([[0.], [3.], [0.], [0.], [0.], [0.], [0.], [0.], [40.]])
high = np.array([[30000.], [12.], [2000.], [180.], [16.], [6.], [20000.], [60.], [100.]])

def project_box(x, low, high):
    return np.minimum(np.maximum(x, low), high)

In [18]:
def maximize(x0, w, X_mean, X_std, α=0.1, steps=1000):
    x = x0.copy()
    w_features = w[:-1, :]
    
    for i in range(steps):
        x_norm = (x.T - X_mean) / X_std
        g_norm = w_features.copy()
        g_original = g_norm / X_std.T
        x_new = x + α * g_original
        x = project_box(x_new, low, high)
        
        if i % 200 == 0:
            satisfaction = (x_norm @ w_features + w[-1, :]) * 10.0
            print(f"Step {i}: Satisfaction = {satisfaction[0,0]:.3f}")
    
    return x

In [21]:
x0 = X_raw_clean.mean(0, keepdims=True).T
x_optimal = maximize(x0, w, X_mean, X_std, α=0.1, steps=1000)

# Calculate final satisfaction score
x_optimal_norm = (x_optimal.T - X_mean) / X_std
x_optimal_with_bias = np.hstack([x_optimal_norm, np.ones((1, 1))])
final_satisfaction = (x_optimal_with_bias @ w) * 10.0

print(f"\n=== RESULTS ===")
print(f"Optimal satisfaction score: {final_satisfaction[0,0]:.3f}")
print(f"Actual maximum in data: {y_score.max():.3f}")

print("\nOptimal parameters:")
labels = ["steps", "time_in_bed_hours", "active_cals", "exercise_min", 
          "nonsedentary_hours", "daylight_hours", "distance_m", "flights", "resting_hr"]
for name, val in zip(labels, x_optimal.flatten()):
    print(f"{name:>20}: {val:.3f}")

Step 0: Satisfaction = 5.458
Step 200: Satisfaction = 6.126
Step 400: Satisfaction = 6.793
Step 600: Satisfaction = 7.460
Step 800: Satisfaction = 7.696

=== RESULTS ===
Optimal satisfaction score: 7.794
Actual maximum in data: 10.000

Optimal parameters:
               steps: 9921.518
   time_in_bed_hours: 9.846
         active_cals: 0.000
        exercise_min: 57.897
  nonsedentary_hours: 14.154
      daylight_hours: 1.151
          distance_m: 7403.819
             flights: 8.139
          resting_hr: 52.737


In [22]:
# Show what actually worked best in the data
best_day_idx = np.argmax(y_score)
best_actual_satisfaction = y_score[best_day_idx, 0]
best_actual_params = X_raw_clean[best_day_idx, :]

print(f"\n=== ACTUAL BEST DAY FROM DATA ===")
print(f"Satisfaction: {best_actual_satisfaction:.3f}")
for name, val in zip(labels, best_actual_params):
    print(f"{name:>20}: {val:.3f}")


=== ACTUAL BEST DAY FROM DATA ===
Satisfaction: 10.000
               steps: 10964.000
   time_in_bed_hours: 8.517
         active_cals: 2.101
        exercise_min: 0.000
  nonsedentary_hours: 15.483
      daylight_hours: 1.675
          distance_m: 8523.000
             flights: 14.000
          resting_hr: 52.000
